# Choosing the Index Surgery

Before an index surgery can be defined, "the surgery" has to be unambiguous, so this notebook
counts the distinct calendar days carrying a surgical procedure for each patient in the surgery
and Fitbit cohort. **Patients with one surgery-day need no index rule at all; the shape of the
tail is what says whether "first", "last" or "longest admission" is a consequential choice and
how many patients it moves.** Every other notebook takes `index_rule="first"` until this one
settles it.

**Governance.** Every query aggregates inside BigQuery and returns counts only, so no
participant-level rows enter the notebook. Unsuppressed figures stay git-ignored while
`*_suppressed.png` is committable, and the small-cell threshold must be confirmed against current
All of Us policy before anything is published.

## Shared Imports and Methods

Imports, the CDR and cache resolution, the cohort SQL and the whole figure setup, in one cell.
The surgical definition is concept `4301351` plus every descendant, and cohort membership is
Fitbit data plus at least one such procedure, built by `srp_common.cohort_ctes` from the CDR alone
so **the definition lives in version-controlled code rather than in Workbench cohort-builder
state**. The added CTE reduces qualifying rows to a surgery-day count per patient, and the
composed prefix is reused by every query below. The palette is one hue plus text and grid greys,
since the distribution is a single series needing no legend, and `figure_path` derives the
filename from the `suppress` flag so the `_suppressed` suffix cannot claim a suppression that did
not happen.

#### Imports, Cohort SQL and the Plotting Helpers

The repo root is found by walking up from the working directory, so the notebook runs from either
its task directory or the root. **`collapse_small_cells` aggregates the sparse tail rather than
masking it**, because a masked bar next to an unmasked total leaks the masked height by
subtraction.

In [ ]:
import sys
from pathlib import Path

import pandas

_root = Path.cwd().resolve()
while not (_root / "srp_common.py").exists():
    if _root.parent == _root:
        raise RuntimeError(
            "Repo root not found: no srp_common.py at or above "
            f"{Path.cwd().resolve()}. Run this notebook from inside the repository."
        )
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import srp_common

cfg = srp_common.init("surgery_day_distribution")
CDR = cfg.CDR
FIGURES, CACHE_DIR = cfg.FIGURES, cfg.CACHE_DIR
cached_gbq = cfg.cached_gbq

# The surgical concept set and the per-patient surgery-day count. index_surgery already
# carries n_surgery_days, but the queries below name this CTE directly, so it is added
# rather than folded in.
SURGERY_ANCESTOR_CONCEPT_ID = srp_common.SURGERY_ANCESTOR_CONCEPT_ID

# index_surgery already carries n_surgery_days, but the queries below name this CTE
# directly, so it is added rather than folded in.
_SURGERY_DAYS_CTE = """
    surgery_days AS (
        SELECT person_id, COUNT(DISTINCT surgery_date) AS n_surgery_days
        FROM surgical_events
        GROUP BY person_id
    )"""

_COHORT_CTES = cfg.cohort_ctes(extra=_SURGERY_DAYS_CTE)

# FIGURE SETUP. One hue plus greys, validated at or above 3:1 contrast on the light
# surface, and the suppression helpers the plot function calls.
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

SUPPRESS_THRESHOLD = 20

SURFACE = "#fcfcfb"
INK = "#0b0b0b"
INK_MUTED = "#52514e"
SERIES = "#2a78d6"
GRID = "#e3e2df"

_thousands = FuncFormatter(lambda v, _: f"{int(v):,}")

def collapse_small_cells(dist, threshold=SUPPRESS_THRESHOLD):
    """Return (kept_rows, tail_label, tail_patients, n_absorbed)."""
    d = dist.sort_values("n_surgery_days").reset_index(drop=True)
    keep = d.copy()
    tail_patients = 0
    absorbed = 0

    while len(keep) > 1 and (
        tail_patients and tail_patients < threshold
        or keep.iloc[-1]["n_patients"] < threshold
    ):
        tail_patients += int(keep.iloc[-1]["n_patients"])
        absorbed += 1
        keep = keep.iloc[:-1]

    if absorbed == 0:
        return keep, None, 0, 0

    first_absorbed = int(d.iloc[len(keep)]["n_surgery_days"])
    return keep, f"{first_absorbed}+", tail_patients, absorbed

def figure_path(name, suppress):
    return FIGURES / f"{name}{'_suppressed' if suppress else ''}.png"


def plot_surgery_days(dist, suppress=False, logy=False, save_as=None):
    d = dist.sort_values("n_surgery_days").reset_index(drop=True)
    tail_label, tail_patients, n_absorbed = None, 0, 0
    if suppress:
        d, tail_label, tail_patients, n_absorbed = collapse_small_cells(dist)
        d = d.copy()

    xs = list(range(len(d)))
    labels = [str(int(v)) for v in d["n_surgery_days"]]
    heights = [int(v) for v in d["n_patients"]]
    if tail_label:
        xs.append(len(d))
        labels.append(tail_label)
        heights.append(tail_patients)

    shown_total = sum(heights)
    cum_share = [sum(heights[: i + 1]) / shown_total for i in range(len(heights))]

    fig = plt.figure(figsize=(10, 8.2), facecolor=SURFACE)
    gs = fig.add_gridspec(3, 1, height_ratios=[0.5, 1.5, 1.0], hspace=0.55)

    ax_kpi = fig.add_subplot(gs[0])
    ax_kpi.set_facecolor(SURFACE)
    ax_kpi.axis("off")
    tiles = [
        (f"{total_patients:,}", "patients with a\ndated surgery-day"),
        (f"{median_days}", "median surgery-days\nper patient"),
        (f"{single_day / total_patients:.0%}", "have exactly\none surgery-day"),
        (f"{multi_day:,}", "have two or more\n(the ones a rule decides)"),
    ]
    for i, (value, label) in enumerate(tiles):
        x = i / len(tiles) + 0.01
        ax_kpi.text(x, 0.62, value, transform=ax_kpi.transAxes, fontsize=25,
                    fontweight="600", color=INK, va="center", ha="left")
        ax_kpi.text(x, 0.12, label, transform=ax_kpi.transAxes, fontsize=9.5,
                    color=INK_MUTED, va="center", ha="left", linespacing=1.5)

    ax = fig.add_subplot(gs[1])
    ax.set_facecolor(SURFACE)
    bars = ax.bar(xs, heights, width=0.78, color=SERIES, linewidth=0, zorder=3)
    ax.set_title("Patients by number of distinct surgery-days",
                 fontsize=13, fontweight="600", color=INK, loc="left", pad=10)
    ax.set_xlabel("distinct days with a surgical procedure", fontsize=10, color=INK_MUTED)
    ax.set_ylabel("patients", fontsize=10, color=INK_MUTED)
    if logy:
        ax.set_yscale("log")
    ax.yaxis.set_major_formatter(_thousands)
    ax.set_xticks(xs)
    ax.set_xticklabels(labels)
    for rect, height in zip(bars, heights):
        ax.text(rect.get_x() + rect.get_width() / 2, rect.get_height(),
                f"{height:,}", ha="center", va="bottom",
                fontsize=8.5, color=INK_MUTED)

    ax2 = fig.add_subplot(gs[2])
    ax2.set_facecolor(SURFACE)
    ax2.step(xs, cum_share, where="mid", color=SERIES, linewidth=2, zorder=3)
    ax2.set_title("Cumulative share of patients at or below N surgery-days",
                  fontsize=13, fontweight="600", color=INK, loc="left", pad=10)
    ax2.set_xlabel("distinct days with a surgical procedure", fontsize=10, color=INK_MUTED)
    ax2.set_ylabel("share of patients", fontsize=10, color=INK_MUTED)
    ax2.set_ylim(0, 1.03)
    ax2.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:.0%}"))
    ax2.set_xticks(xs)
    ax2.set_xticklabels(labels)
    for level, va in ((0.90, "top"), (0.95, "bottom")):
        ax2.axhline(level, color=GRID, linewidth=1, linestyle="--", zorder=1)
        ax2.text(1.005, level, f"{level:.0%}", transform=ax2.get_yaxis_transform(),
                 va=va, ha="left", fontsize=8, color=INK_MUTED, clip_on=False)

    for a in (ax, ax2):
        a.grid(axis="y", color=GRID, linewidth=1, zorder=0)
        a.set_axisbelow(True)
        for side in ("top", "right"):
            a.spines[side].set_visible(False)
        for side in ("left", "bottom"):
            a.spines[side].set_color(GRID)
        a.tick_params(colors=INK_MUTED, labelsize=9)

    if suppress:
        note = (f"tail aggregated into '{tail_label}' ({n_absorbed} cell(s), "
                f"{tail_patients:,} patients) to meet the {SUPPRESS_THRESHOLD}-patient threshold"
                if tail_label else
                f"no cell below {SUPPRESS_THRESHOLD} patients; nothing aggregated")
    else:
        n_small = int((d["n_patients"] < SUPPRESS_THRESHOLD).sum())
        note = (f"NOT suppressed -- {n_small} cell(s) below {SUPPRESS_THRESHOLD} patients; "
                f"do not share this version")
    fig.text(0.01, 0.005, f"All of Us  |  CDR {CDR}  |  {note}",
             fontsize=8, color=INK_MUTED)

    if save_as:
        out = figure_path(save_as, suppress)
        fig.savefig(out, dpi=150, bbox_inches="tight", facecolor=SURFACE)
        print(f"saved -> {out}"
              + ("   (committable)" if suppress else "   (local only, git-ignored)"))
    return fig

## Part 1 — How Many Surgery-Days a Patient Has

The histogram, the denominators that bound it, and what the two together say about an index rule.
**The histogram is built inside BigQuery, returning one row per distinct surgery-day count rather
than one row per patient**, with running totals added so order statistics are recoverable from the
histogram alone and no per-patient row is ever held. The denominators matter because a patient can
qualify yet contribute no surgery-day, and because rows without a `procedure_datetime` are
invisible to a day-based analysis — a large undated count would undermine any date-based index
rule.

#### The Distribution, Its Denominators, and How Concentrated It Is

The median and 90th percentile are read off the cumulative counts, and the split between one and
several surgery-days isolates the population an index rule actually decides anything for — **that
group's size is the ceiling on how much the choice of rule can matter**. The unsuppressed figure
is the one to read for the decision and stays git-ignored, while the suppressed variant
aggregates the sparse tail and is the version that may be committed or shared.

In [ ]:
distribution_sql = _COHORT_CTES + """
    SELECT
        n_surgery_days,
        COUNT(*) AS n_patients
    FROM surgery_days
    GROUP BY n_surgery_days
    ORDER BY n_surgery_days
"""

surgery_day_dist = cached_gbq(distribution_sql, "surgery_day_distribution")

dist = surgery_day_dist.sort_values("n_surgery_days").reset_index(drop=True)
total_patients = int(dist["n_patients"].sum())
total_surgery_days = int((dist["n_surgery_days"] * dist["n_patients"]).sum())

dist["cum_patients"] = dist["n_patients"].cumsum()
dist["cum_share"] = dist["cum_patients"] / total_patients
dist

# The denominators the histogram has to be read against: a patient can qualify yet
# contribute no surgery-day, and an undated procedure row is invisible here.
qc_sql = _COHORT_CTES + """
    SELECT
        (SELECT COUNT(*) FROM cohort) AS cohort_patients,
        (SELECT COUNT(*) FROM surgery_days) AS patients_with_dated_surgery,
        (SELECT COUNT(*) FROM surgical_events) AS dated_surgical_rows,
        (SELECT COUNT(*) FROM surgical_concept) AS n_surgical_concepts
"""

qc = cached_gbq(qc_sql, "surgery_day_qc").iloc[0]

missing_sql = _COHORT_CTES + f"""
    SELECT COUNT(*) AS rows_missing_datetime
    FROM `{CDR}`.procedure_occurrence AS po
    WHERE po.person_id IN (SELECT person_id FROM cohort)
      AND po.procedure_concept_id IN (SELECT concept_id FROM surgical_concept)
      AND po.procedure_datetime IS NULL
"""

rows_missing_datetime = int(
    cached_gbq(missing_sql, "surgery_rows_missing_datetime").iloc[0, 0]
)

print(f"cohort patients (index tables)      : {qc['cohort_patients']:>9,}")
print(f"patients with >=1 dated surgery-day : {total_patients:>9,}")
print(f"  ... contributing surgery-days     : {total_surgery_days:>9,}")
print(f"surgical rows lacking a datetime    : {rows_missing_datetime:>9,}")
print(f"distinct surgical concepts in set   : {qc['n_surgical_concepts']:>9,}")

# Order statistics straight off the cumulative counts, so no per-patient row is needed.
def quantile_from_hist(q):
    """Smallest surgery-day count at which the cumulative share reaches q."""
    idx = (dist["cum_patients"] >= q * total_patients).idxmax()
    return int(dist.loc[idx, "n_surgery_days"])


median_days = quantile_from_hist(0.50)
p90_days = quantile_from_hist(0.90)
max_days = int(dist["n_surgery_days"].max())

print(f"surgery-days per patient  median={median_days}  p90={p90_days}  max={max_days}")

single_day = int(dist.loc[dist["n_surgery_days"] == 1, "n_patients"].sum())
multi_day = total_patients - single_day

print(f"exactly one surgery-day   : {single_day:>9,}  ({single_day / total_patients:6.1%})")
print(f"two or more surgery-days  : {multi_day:>9,}  ({multi_day / total_patients:6.1%})")
print()
print("--> The multi-day share is the population an index-surgery rule actually decides.")

# Both variants are written: unsuppressed to read, suppressed to share.
fig = plot_surgery_days(dist, suppress=False, save_as="surgery_days")
plot_surgery_days(dist, suppress=True, save_as="surgery_days")

if max(dist["n_patients"]) > 50 * min(dist["n_patients"]):
    print("\nTail is orders of magnitude below the mode -- "
          "rerun with logy=True to read it: plot_surgery_days(dist, logy=True)")